# SALT3 Colab Runbook

This notebook is an execution checklist for the SALT3 workflow. It is intentionally lightweight: the real code lives in `salt3_common.py`, and model state lives in Google Drive under `/content/drive/MyDrive/SALT3`.

## Runtime model

Each Colab notebook starts from a clean runtime. It can import `.py` code from Drive, but it cannot see variables from another notebook. Durable dependencies must be saved as files under `/content/drive/MyDrive/SALT3`.

Use this pattern in every notebook: mount Drive, add the SALT3 code folder to `sys.path`, load config/artifacts from Drive, then write this notebook's outputs back to Drive.

In [1]:
from pathlib import Path
import shutil
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not running in Colab or Drive is already unavailable:', exc)

PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
CODE_DIR = PROJECT_ROOT / 'code'
CODE_DIR.mkdir(parents=True, exist_ok=True)

# If salt3_common.py is uploaded beside this notebook, copy it into Drive code/.
local_common = Path('/content/salt3_common.py')
if local_common.exists():
    shutil.copy2(local_common, CODE_DIR / 'salt3_common.py')

sys.path.insert(0, str(CODE_DIR))
sys.path.insert(0, '/content')
print('SALT3 project root:', PROJECT_ROOT)
print('SALT3 code dir    :', CODE_DIR)

Mounted at /content/drive
SALT3 project root: /content/drive/MyDrive/SALT3
SALT3 code dir    : /content/drive/MyDrive/SALT3/code


## Folder contract

The expected Drive layout is:

```text
/content/drive/MyDrive/SALT3/
  code/salt3_common.py
  init/<INIT_NAME>/model/
  runs/<RUN_NAME>/run_config.json
  runs/<RUN_NAME>/checkpoints/
  runs/<RUN_NAME>/final_model/
  runs/<RUN_NAME>/metrics.jsonl
  eval/<EVAL_NAME>/
  datasets/
```

A run is resumed only from its own `runs/<RUN_NAME>/checkpoints/`. A continuation run starts from another model artifact but writes into a new run folder.

In [2]:
for child in ['code', 'init', 'runs', 'eval', 'datasets']:
    (PROJECT_ROOT / child).mkdir(parents=True, exist_ok=True)
print('Ready folders:')
for child in sorted(PROJECT_ROOT.iterdir()):
    print(' -', child)

Ready folders:
 - /content/drive/MyDrive/SALT3/code
 - /content/drive/MyDrive/SALT3/datasets
 - /content/drive/MyDrive/SALT3/eval
 - /content/drive/MyDrive/SALT3/init
 - /content/drive/MyDrive/SALT3/runs


## Run modes

Use `02_train_cpt_run.ipynb` with one of these modes:

```python
MODE = 'new'
RUN_NAME = 'simple_cpt_400k_lr1e-4'
BASE_MODEL_REF = 'init/videberta_salt_init_v1/model'

MODE = 'resume'
RUN_NAME = 'simple_cpt_400k_lr1e-4'

MODE = 'continue'
RUN_NAME = 'simple_cpt_2M_continue_lr5e-5'
BASE_MODEL_REF = 'runs/simple_cpt_2M/final_model'
```